In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

# --- B1. Problem Setup: Canonical Encodings ---

# Canonical encodings for digits 0-9 (a, b, c, d, e, f, g)
# Segment Map:
#    a
#  f   b
#    g
#  e   c
#    d
CANONICAL_PATTERNS = np.array([
    [1, 1, 1, 1, 1, 1, 0],  # 0
    [0, 1, 1, 0, 0, 0, 0],  # 1
    [1, 1, 0, 1, 1, 0, 1],  # 2
    [1, 1, 1, 1, 0, 0, 1],  # 3
    [0, 1, 1, 0, 0, 1, 1],  # 4
    [1, 0, 1, 1, 0, 1, 1],  # 5
    [1, 0, 1, 1, 1, 1, 1],  # 6
    [1, 1, 1, 0, 0, 0, 0],  # 7
    [1, 1, 1, 1, 1, 1, 1],  # 8
    [1, 1, 1, 1, 0, 1, 1]   # 9
], dtype=np.float32)

NUM_CLASSES = 10
P_FLIP = 0.05
# NOTE: The search space is now limited to one combination (H1=16, H2=16, LR=1e-2)
# to prevent kernel crashes. 50 epochs is used for the K-Fold study.
EPOCHS_FOR_STUDY = 50
FINAL_EPOCHS = 200 # Used for the final robustness check (B4)
SEED = 42

# Create one-hot targets
Y_CANONICAL = np.eye(NUM_CLASSES, dtype=np.float32)

In [2]:
# --- B2. Training & Augmentation: Data Generation ---

def generate_augmented_data(n_variants_per_digit, p_flip=P_FLIP, seed=SEED):
    """
    Creates noisy variants of the canonical patterns by flipping each bit
    with probability p_flip.
    """
    np.random.seed(seed)
    X_train_list = []
    Y_train_list = []

    for digit in range(NUM_CLASSES):
        base_pattern = CANONICAL_PATTERNS[digit]

        for _ in range(n_variants_per_digit):
            noisy_pattern = np.copy(base_pattern)

            # Generate a mask of bits to flip (1 where flip occurs)
            flip_mask = np.random.rand(7) < p_flip

            # Flip the bits (0 -> 1, 1 -> 0)
            noisy_pattern[flip_mask] = 1.0 - noisy_pattern[flip_mask]

            X_train_list.append(noisy_pattern)
            Y_train_list.append(Y_CANONICAL[digit])

    X_train = np.array(X_train_list, dtype=np.float32)
    Y_train = np.array(Y_train_list, dtype=np.float32)

    return X_train, Y_train

# Generate 50 noisy variants per digit (500 total samples)
X_DATA, Y_DATA = generate_augmented_data(n_variants_per_digit=50)

In [3]:
# --- B1/B2. Neural Network Model Definition ---

def create_model(h1_width, h2_width, learning_rate):
    """
    Defines the [7 -> H1 -> H2 -> 10] architecture.
    ReLU in hidden layers, Softmax output.
    """
    tf.random.set_seed(SEED)
    model = Sequential([
        Dense(h1_width, activation='relu', input_shape=(7,)),  # H1
        Dense(h2_width, activation='relu'),                     # H2
        Dense(NUM_CLASSES, activation='softmax')               # Output
    ])

    # Loss: cross-entropy, Optimizer: Adam
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [4]:
# --- B3. Hyperparameter Study & K-Fold Cross-Validation ---

def run_kfold_and_evaluate(X, Y, hparams, k_splits=5):
    """
    Performs k-fold cross-validation for a given set of hyperparameters
    and reports macro-averaged metrics.
    """
    kf = KFold(n_splits=k_splits, shuffle=True, random_state=SEED)

    # Store metrics for each fold
    acc_scores, prec_scores, rec_scores, f1_scores = [], [], [], []
    loss_history = []

    h1_width, h2_width, learning_rate = hparams
    print(f"--- Testing H1={h1_width}, H2={h2_width}, LR={learning_rate:.4f}, Epochs={EPOCHS_FOR_STUDY} ---")

    fold = 1
    for train_index, val_index in kf.split(X):
        # Create model for the fold
        model = create_model(h1_width, h2_width, learning_rate)

        X_train, X_val = X[train_index], X[val_index]
        Y_train, Y_val = Y[train_index], Y[val_index]

        # Train for reduced epochs
        history = model.fit(X_train, Y_train,
                            epochs=EPOCHS_FOR_STUDY,
                            batch_size=32,
                            verbose=0,
                            validation_data=(X_val, Y_val))

        # Record loss for plotting (using the first fold's loss history)
        if fold == 1:
            loss_history = history.history['loss']

        # Evaluate on the validation fold
        Y_pred_one_hot = model.predict(X_val, verbose=0)
        Y_pred = np.argmax(Y_pred_one_hot, axis=1)
        Y_true = np.argmax(Y_val, axis=1)

        # Calculate metrics (macro average for multi-class classification)
        acc = accuracy_score(Y_true, Y_pred)
        # Setting zero_division=0 to handle cases where a class has no predictions in a fold
        prec = precision_score(Y_true, Y_pred, average='macro', zero_division=0)
        rec = recall_score(Y_true, Y_pred, average='macro', zero_division=0)
        f1 = f1_score(Y_true, Y_pred, average='macro', zero_division=0)

        acc_scores.append(acc)
        prec_scores.append(prec)
        rec_scores.append(rec)
        f1_scores.append(f1)

        fold += 1

    # Calculate means and standard deviations
    results = {
        'H1': h1_width,
        'H2': h2_width,
        'LR': learning_rate,
        'Accuracy_mean': np.mean(acc_scores),
        'Accuracy_std': np.std(acc_scores),
        'Precision_mean': np.mean(prec_scores),
        'Precision_std': np.std(prec_scores),
        'Recall_mean': np.mean(rec_scores),
        'Recall_std': np.std(rec_scores),
        'F1_mean': np.mean(f1_scores),
        'F1_std': np.std(f1_scores),
        'Loss_history': loss_history,
        'Val_Accuracy': np.mean(acc_scores), # Mean Accuracy used for finding best combo
    }

    return results


def run_hyperparameter_study(X, Y):
    """
    Iterates through only one hyperparameter combination for safety.
    """
    # Hyperparameter search space is REDUCED to prevent crashes
    H_WIDTHS = [16]  # Reduced from [8, 16, 32]
    LEARNING_RATES = [1e-2] # Reduced from [5e-3, 1e-2, 5e-2]

    all_results = []
    best_result = None

    # Only 1 combination will be tested (16, 16, 0.01)
    print("Starting MINIMAL Hyperparameter Study (1 combination)...")

    for lr in LEARNING_RATES:
        for h1 in H_WIDTHS:
            for h2 in H_WIDTHS:
                hparams = (h1, h2, lr)
                result = run_kfold_and_evaluate(X, Y, hparams)
                all_results.append(result)

                # Track the result
                best_result = result

    print("\n--- Minimal Study Complete ---")
    print("\nBest (and only) Hyperparameter Combination Tested:")
    print(f"H1: {best_result['H1']}, H2: {best_result['H2']}, LR: {best_result['LR']:.4f}")

    print("\nK-Fold Cross-Validation Metrics for Best Combination (k=5):")
    # Note: These metrics are based on the reduced 50 epochs run.
    print(f"Accuracy: {best_result['Accuracy_mean']:.4f} \u00B1 {best_result['Accuracy_std']:.4f}")
    print(f"Precision: {best_result['Precision_mean']:.4f} \u00B1 {best_result['Precision_std']:.4f}")
    print(f"Recall: {best_result['Recall_mean']:.4f} \u00B1 {best_result['Recall_std']:.4f}")
    print(f"F1 (macro): {best_result['F1_mean']:.4f} \u00B1 {best_result['F1_std']:.4f}")

    print(f"\nLoss History (First Fold of Best Model, {EPOCHS_FOR_STUDY} Epochs - B2 plot):")
    # Displaying the first 5 and last 5 loss values to show convergence trend
    print([f'{l:.4f}' for l in best_result['Loss_history'][:5]] + ['...',] + [f'{l:.4f}' for l in best_result['Loss_history'][-5:]])

    return best_result

In [5]:
# --- B4. Robustness Check (Optional for clean patterns) ---

def check_robustness(h1, h2, lr, X_train, Y_train):
    """
    Trains the best model once on the full 200 epochs and checks performance
    on canonical (non-noisy) data.
    """
    print("\n--- Robustness Check on Canonical (Clean) Patterns using 200 Epochs ---")

    # Train the final model on all augmented data for the required 200 epochs
    model = create_model(h1, h2, lr)
    print("Training final model (200 epochs)...")
    # Increased verbosity for the final, important run
    model.fit(X_train, Y_train, epochs=FINAL_EPOCHS, batch_size=32, verbose=1)

    # Evaluate on the perfectly clean canonical patterns
    Y_pred_one_hot = model.predict(CANONICAL_PATTERNS, verbose=0)
    Y_pred = np.argmax(Y_pred_one_hot, axis=1)
    Y_true = np.arange(NUM_CLASSES)

    correct_predictions = Y_pred == Y_true

    print("\nCanonical Pattern Predictions:")
    confused_digits = []
    for true_digit, predicted_digit, is_correct in zip(Y_true, Y_pred, correct_predictions):
        status = "CORRECT" if is_correct else f"INCORRECT (Predicted: {predicted_digit})"
        print(f"Digit {true_digit}: {status}")
        if not is_correct:
            confused_digits.append(true_digit)

    if not confused_digits:
        print("\nThe network correctly classified all canonical (clean) patterns.")
    else:
        print(f"\nThe network failed to classify the following clean digits: {confused_digits}")


if __name__ == '__main__':
    # Ensure TensorFlow is not noisy
    tf.get_logger().setLevel('ERROR')

    # Main execution for B3 (using reduced epochs)
    best_hparams = run_hyperparameter_study(X_DATA, Y_DATA)

    # B4: Robustness check using the *best* parameters found from the study,
    # but trained for the required FINAL_EPOCHS (200).
    check_robustness(
        best_hparams['H1'],
        best_hparams['H2'],
        best_hparams['LR'],
        X_DATA,
        Y_DATA
    )

Starting MINIMAL Hyperparameter Study (1 combination)...
--- Testing H1=16, H2=16, LR=0.0100, Epochs=50 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/lo


--- Minimal Study Complete ---

Best (and only) Hyperparameter Combination Tested:
H1: 16, H2: 16, LR: 0.0100

K-Fold Cross-Validation Metrics for Best Combination (k=5):
Accuracy: 0.8540 ± 0.0367
Precision: 0.8583 ± 0.0325
Recall: 0.8531 ± 0.0388
F1 (macro): 0.8493 ± 0.0366

Loss History (First Fold of Best Model, 50 Epochs - B2 plot):
['2.1787', '1.8010', '1.3348', '0.9115', '0.6388', '...', '0.3476', '0.3444', '0.3466', '0.3414', '0.3455']

--- Robustness Check on Canonical (Clean) Patterns using 200 Epochs ---
Training final model (200 epochs)...
Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - accuracy: 0.1520 - loss: 2.2744
Epoch 2/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3395 - loss: 1.9810   
Epoch 3/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4789 - loss: 1.5252 
Epoch 4/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6419 - loss: 1.0885 
Epoch 5/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7820 - loss: 0.8018 
Epoch 6/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8232 - loss: 0.6664 
Epoch 7/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8305 - loss: 0.5927 
Epoch 8/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8444 - loss: 0.5523 
Epoch 9/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8508 - loss: 0.5278 
Epoch 10/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8604 - loss: 0.5051 
Epoch 11/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8581 - loss: 0.4820 
Epoch 12/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy